# Drifting Resonators

This notebook demonstrates how to use the OOP API for analyzing real ring-down measurement data,
with the Q workflow recommended after the 2026-08-18 Q-estimation investigation.

The headline number is **`Q_selected`**: the pipeline classifies each record from its own
diagnostics and picks the estimator whose assumptions actually hold. On real records that is
almost always **`Q_demod`**, the incoherent segmented-demodulation estimate
(`ringdownanalysis.demod.SegmentedDemodEstimator`), which is immune to the 0.3–1.1 mHz frequency
drift that biases every phase-coherent fit and which models the ambient-driven plateau explicitly.

The coherent estimates (`Q_nls`, `Q_dft`, `Q_profile`) are still computed and shown here, but as
diagnostics: the pipeline demotes them to non-valid when the measured drift breaks phase coherence
(`coherence_gate_fired`) or when they contradict the measured envelope slope
(`envelope_mismatch`). See the "Which Q should I trust?" section of the README, and
[`20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb`](20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb)
for the estimator's validation on synthetics with known truth.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ringdownanalysis import RingDownAnalyzer, SegmentedDemodEstimator, fit_nonlinear_damping

In [ ]:
# Apply consistent plotting style
from ringdownanalysis import plots

plots.apply_plotting_style()

## Initialize Analyzer

The defaults are the recommended configuration: the segmented-demodulation estimator splits the
record into about 96 segments, detects the driven plateau, and fits the floor-corrected decay.
Every estimator can be swapped or retuned if a record needs it.


In [ ]:
# Create analyzer with default estimators
analyzer = RingDownAnalyzer()

# Or retune the segmented-demodulation estimator, e.g. longer segments for a
# low-frequency resonance, or a stricter plateau cutoff:
# demod_est = SegmentedDemodEstimator(seg_duration=120.0, floor_threshold=4.0)
# analyzer = RingDownAnalyzer(demod_estimator=demod_est)

# The coherent frequency estimators can still be customized (they drive f_nls/f_dft):
# from ringdownanalysis import NLSFrequencyEstimator, DFTFrequencyEstimator
# nls_est = NLSFrequencyEstimator(tau_known=None)
# dft_est = DFTFrequencyEstimator(window='kaiser', kaiser_beta=9.0)
# analyzer = RingDownAnalyzer(nls_estimator=nls_est, dft_estimator=dft_est)

## Find Data Files

Measurement files live in **subdirectories** of `data/` (for example `data/MTS/*.mat` and
`data/ODIN/*.csv.zip`), so a flat `data/*.csv` glob finds nothing — discovery has to recurse.

`RingDownDataLoader` reads plain `.csv` and `.mat` phasemeter exports. The zipped ODIN Phasemeter
exports (`.csv.zip`) are loaded with `mokutools` in the dedicated ODIN notebooks and are listed but
skipped here.

Records run to several GB. This walkthrough analyzes the first `MAX_FILES` files below the
`MAX_FILE_SIZE_MB` cap so it stays interactive; raise both to cover everything (or use
`0.2_batch-analysis.ipynb`, which is built for that).


In [ ]:
MAX_FILE_SIZE_MB = 250.0  # skip very large records in this walkthrough
MAX_FILES = 3

data_dir = (Path("..") / "data").resolve()
print(f"Data directory: {data_dir}  (exists: {data_dir.is_dir()})")

csv_files = sorted(p for p in data_dir.rglob("*.csv") if p.is_file())
mat_files = sorted(p for p in data_dir.rglob("*.mat") if p.is_file())
zip_files = sorted(p for p in data_dir.rglob("*.csv.zip") if p.is_file())

loadable = csv_files + mat_files
print(f"Found {len(csv_files)} CSV and {len(mat_files)} MAT files loadable by RingDownDataLoader")
print(f"Found {len(zip_files)} zipped ODIN exports (.csv.zip) — load these with mokutools instead")

size_mb = {p: p.stat().st_size / 1e6 for p in loadable}
small_enough = [p for p in loadable if size_mb[p] <= MAX_FILE_SIZE_MB]
selected = small_enough[:MAX_FILES]

print(f"\n{len(loadable)} loadable file(s):")
for p in loadable:
    if p in selected:
        mark = "analyze"
    elif p in small_enough:
        mark = f"skip (beyond MAX_FILES={MAX_FILES})"
    else:
        mark = f"skip (> {MAX_FILE_SIZE_MB:.0f} MB)"
    print(f"  [{mark:>28}] {p.relative_to(data_dir)}  ({size_mb[p]:.0f} MB)")

if not selected:
    print(
        "\nNothing selected. Raise MAX_FILE_SIZE_MB / MAX_FILES, or add smaller "
        ".csv / .mat records under data/."
    )

## Analyze Each File

The analyzer performs the complete pipeline:
1. Load data and validate finite time/phase values
2. Estimate tau from the full record, cross-checked against the envelope tau
   (`tau_est_low_confidence` flags a disagreement, and the crop then uses the envelope value)
3. Crop data to `max_tau_multiplier * tau_est` (default: 1x tau), never below one envelope tau
4. Run the segmented-demodulation estimator on the full record: `Q_demod` with a block-bootstrap
   CI, plus the plateau level, the frequency drift, and `coherence_ratio` = |drift| × tau
5. Estimate frequency using NLS and DFT, and Q with the coherent profile likelihood — demoted to
   non-valid when the drift gate (`coherence_ratio` > 0.01) or the envelope-mismatch gate fires
6. Select the trustworthy answer: `Q_selected` with its `source`, `regime` and agreement ratio
7. Estimate noise parameters and report a plug-in frequency uncertainty diagnostic for the crop


In [ ]:
def fmt(value, spec=".4g"):
    """Format a possibly-missing estimate."""
    return format(value, spec) if value is not None and np.isfinite(value) else "—"


results = []

for filepath in selected:
    try:
        print(f"\nProcessing {filepath.name}...")
        r = analyzer.analyze_file(str(filepath))
        results.append(r)

        print(f"  Sampling frequency: {r['fs']:.2f} Hz")
        print(f"  Record duration: {r['T']:.1f} s ({r['N']} samples)")
        print(f"  Estimated tau: {r['tau_est']:.2f} s (low confidence: {r['tau_est_low_confidence']})")
        print(f"  Cropped to: {r['T_crop']:.2f} s ({r['N_crop']} samples), source {r['tau_crop_source']}")

        print("  --- recommended Q ---")
        print(
            f"  Q_selected: {fmt(r['Q_selected'], '.4e')}"
            f"  [source {r['Q_selected_source']}, regime {r['Q_selected_regime']},"
            f" status {r['Q_selected_status']}]"
        )
        if r["Q_selected_reasons"]:
            print(f"    reasons: {', '.join(r['Q_selected_reasons'])}")
        if r["Q_selected_agreement_ratio"] is not None:
            print(f"    cross-estimator agreement ratio: {r['Q_selected_agreement_ratio']:.3f}")

        print("  --- segmented demodulation ---")
        ci = r["Q_demod_ci95"]
        ci_text = f"[{ci[0]:.3e}, {ci[1]:.3e}]" if ci is not None else "—"
        print(f"  Q_demod: {fmt(r['Q_demod'], '.4e')} 95% CI {ci_text} ({r['Q_demod_status']})")
        if r["Q_demod_reasons"]:
            print(f"    reasons: {', '.join(r['Q_demod_reasons'])}")
        print(f"  tau_demod: {fmt(r['tau_demod'], '.1f')} s, f_demod: {fmt(r['f_demod'], '.6f')} Hz")
        print(
            f"  segments: {r['Q_demod_n_segments']} of {r['Q_demod_seg_duration']:.0f} s, "
            f"{int(np.count_nonzero(r['Q_demod_decay_mask']))} in the decay fit"
        )
        print(
            f"  plateau: {fmt(r['Q_demod_plateau_amplitude'], '.3g')} "
            f"(detected: {r['Q_demod_plateau_detected']})"
        )
        drift_mhz = r["Q_demod_drift_hz"] * 1e3 if r["Q_demod_drift_hz"] is not None else None
        print(f"  drift over decay: {fmt(drift_mhz, '+.3f')} mHz")
        print(
            f"  coherence_ratio: {fmt(r['coherence_ratio'], '.3g')} "
            f"(coherent fits need < 0.01; gate fired: {r['coherence_gate_fired']})"
        )

        print("  --- coherent estimators (diagnostics) ---")
        print(f"  NLS frequency: {r['f_nls']:.6f} Hz, DFT frequency: {r['f_dft']:.6f} Hz")
        print(
            f"  Q_profile: {fmt(r['Q_profile'], '.4e')} ({r['Q_profile_status']}), "
            f"raw {fmt(r['Q_profile_raw'], '.4e')}"
        )
        print(f"  Q_nls: {fmt(r['Q_nls'], '.4e')} ({r['Q_nls_status']}), raw {fmt(r['Q_nls_raw'], '.4e')}")
        print(f"  Q_envelope: {fmt(r['Q_envelope'], '.4e')} ({r['Q_envelope_status']})")
        print(f"  Plug-in frequency uncertainty std: {r['plugin_crlb_std_f']:.6e} Hz")
        print(f"  DFT fallback used: {r['dft_used_fallback']}")
    except Exception as e:
        print(f"  Error: {e}")

## Visualize Results

Three views of the first record: the raw phase with the analyzed crop, the demodulated segment
amplitudes with the plateau level and the fitted decay, and the segment frequency versus time —
the drift measurement that decides whether any coherent estimator can be trusted.


In [ ]:
if len(results) > 0:
    # Plot first result
    r = results[0]
    demod = r["Q_demod_result"]
    mask = demod.decay_mask

    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    # Raw phase with the analyzed crop
    ax = axes[0]
    step = max(1, len(r["t"]) // 50000)
    ax.plot(r["t"][::step], r["data"][::step], "b-", alpha=0.5, label="Original")
    step_crop = max(1, len(r["t_crop"]) // 50000)
    ax.plot(
        r["t_crop"][::step_crop], r["data_cropped"][::step_crop], "r-", alpha=0.7, label="Analyzed crop"
    )
    ax.axvline(
        r["T_crop"], color="r", linestyle="--", label=f"T_crop = {r['T_crop']:.1f} s"
    )
    tau_3x = 3.0 * r["tau_est"]
    if tau_3x <= r["T"]:
        ax.axvline(tau_3x, color="g", linestyle=":", label=f"3×τ reference = {tau_3x:.1f} s")
    else:
        ax.plot([], [], " ", label=f"3×τ = {tau_3x:.1f} s (beyond record)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Phase (cycles)")
    ax.set_title(f"{r['filename']}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Demodulated segment amplitudes: plateau level and floor-corrected decay fit
    ax = axes[1]
    ax.semilogy(demod.t_mid, demod.amplitude, ".", ms=5, color="0.6", label="Segment amplitude")
    ax.semilogy(demod.t_mid[mask], demod.amplitude[mask], ".", ms=5, color="C0", label="Decay region (fit)")
    if demod.plateau_detected and demod.plateau_amplitude is not None:
        ax.axhline(
            demod.plateau_amplitude,
            color="r",
            linestyle=":",
            label=f"Plateau ≈ {demod.plateau_amplitude:.3g}",
        )
    if demod.log_slope is not None:
        t_fit = demod.t_mid[mask]
        ax.semilogy(
            t_fit,
            np.exp(demod.log_intercept + demod.log_slope * t_fit),
            "g-",
            linewidth=2,
            label=f"Fit: τ = {demod.tau:.0f} s, Q = {demod.Q:.3e}",
        )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude (cycles)")
    ax.set_title(f"Segmented demodulation ({demod.n_segments} segments of {demod.seg_duration:.0f} s)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Segment frequency: the drift that gates the coherent estimators
    ax = axes[2]
    f_ref = demod.f_mean if demod.f_mean is not None else float(np.median(demod.f_seg))
    ax.plot((demod.t_mid - demod.t_mid[0]), (demod.f_seg - f_ref) * 1e3, ".", ms=5, color="0.6",
            label="All segments")
    ax.plot((demod.t_mid[mask] - demod.t_mid[0]), (demod.f_seg[mask] - f_ref) * 1e3, ".", ms=5,
            color="C0", label="Decay region")
    ax.axhline(0.0, color="k", linestyle=":", linewidth=1)
    drift_text = (
        f"drift = {demod.drift_hz * 1e3:+.3f} mHz, coherence ratio = {demod.coherence_ratio:.3g}"
        if demod.drift_hz is not None
        else "drift unavailable"
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(f"f − {f_ref:.6f} Hz  (mHz)")
    ax.set_title(f"Frequency drift: {drift_text}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Summary Statistics

Per-record summary of the recommended Q and its diagnostics, with the gated coherent values
alongside for comparison. Frequency estimates are summarized separately: NLS and DFT remain the
frequency estimators of record even where their Q is not trustworthy.


In [ ]:
if len(results) > 0:
    summary = pd.DataFrame(
        [
            {
                "file": Path(r["filename"]).name,
                "T (s)": r["T"],
                "tau_demod (s)": r["tau_demod"],
                "Q_selected": r["Q_selected"],
                "source": r["Q_selected_source"],
                "regime": r["Q_selected_regime"],
                "status": r["Q_selected_status"],
                "Q_demod": r["Q_demod"],
                "Q_demod_ci_low": r["Q_demod_ci95"][0] if r["Q_demod_ci95"] else np.nan,
                "Q_demod_ci_high": r["Q_demod_ci95"][1] if r["Q_demod_ci95"] else np.nan,
                "Q_profile (gated)": r["Q_profile"],
                "Q_profile_raw": r["Q_profile_raw"],
                "Q_envelope": r["Q_envelope"],
                "coherence_ratio": r["coherence_ratio"],
                "gate_fired": r["coherence_gate_fired"],
                "plateau": r["Q_demod_plateau_amplitude"] if r["Q_demod_plateau_detected"] else np.nan,
                "drift (mHz)": (r["Q_demod_drift_hz"] or np.nan) * 1e3,
                "f_demod (Hz)": r["f_demod"],
                "f_nls (Hz)": r["f_nls"],
                "f_dft (Hz)": r["f_dft"],
            }
            for r in results
        ]
    )
    display(summary)

    f_nls_all = np.array([r["f_nls"] for r in results])
    f_dft_all = np.array([r["f_dft"] for r in results])
    q_selected = np.array([r["Q_selected"] for r in results if r["Q_selected"] is not None], dtype=float)

    print("Summary Statistics:")
    print(f"  Number of files: {len(results)}")
    print(f"  Records with a valid Q_selected: {len(q_selected)} of {len(results)}")
    if len(q_selected) > 0:
        print(f"  Q_selected mean: {np.mean(q_selected):.4e}")
        print(f"  Q_selected std: {np.std(q_selected):.4e}")
        if len(q_selected) > 1:
            print(f"  Q_selected max/min: {q_selected.max() / q_selected.min():.3f}")
    print(f"  NLS mean: {np.mean(f_nls_all):.9f} Hz")
    print(f"  NLS std: {np.std(f_nls_all):.6e} Hz")
    print(f"  DFT mean: {np.mean(f_dft_all):.9f} Hz")
    print(f"  DFT std: {np.std(f_dft_all):.6e} Hz")
    print(f"  Mean |f_NLS − f_DFT|: {np.mean(np.abs(f_nls_all - f_dft_all)):.6e} Hz")

## Amplitude-Dependent Damping

Real resonators can damp faster at high amplitude, in which case there is no single "true Q" and
any single-exponential estimate is a window-dependent average. The demodulated segments make this
measurable: `Q_demod_vs_amplitude` gives the local Q per amplitude band, and
`fit_nonlinear_damping` fits the amplitude laws $1/\tau(A) = 1/\tau_0 + \beta A$ and
$f(A) = f_\mathrm{zero} + \mathrm{pull} \cdot A$, so records can be compared at matched amplitude
via `nl.q_at(A)` instead of by their window-averaged Q.

In [ ]:
if len(results) == 0:
    print("No results available. Run the processing cell above first.")
else:
    r = results[0]
    demod = r["Q_demod_result"]
    bands = demod.q_vs_amplitude

    print(f"{Path(r['filename']).name}: amplitude-banded local Q")
    for b in bands:
        print(
            f"  A = {b.amplitude_low:8.3g}–{b.amplitude_high:8.3g} cycles "
            f"({b.n_segments:3d} segments): tau = {b.tau:8.1f} s, Q = {b.Q:.3e}"
        )
    if len(bands) > 1:
        q_band = np.array([b.Q for b in bands])
        print(f"  local-Q spread across bands: {q_band.max() / q_band.min():.2f}×")

    nl = fit_nonlinear_damping(demod)
    print(f"\nnonlinear-damping fit: {nl.status} ({', '.join(nl.reasons) or 'no warnings'})")
    if nl.tau0 is not None:
        print(f"  tau0 = {nl.tau0:.1f} ± {fmt(nl.tau0_stderr, '.1f')} s   (zero-amplitude decay time)")
        print(f"  beta = {nl.beta:.3e} ± {fmt(nl.beta_stderr, '.1e')}   (1/tau(A) = 1/tau0 + beta·A)")
        print(f"  Q0   = {nl.Q0:.4e}   (zero-amplitude Q)")
        print(
            f"  f_zero = {nl.f_zero:.6f} Hz, pull = {nl.f_pull:.3e} Hz/cycle "
            f"± {fmt(nl.f_pull_stderr, '.1e')}"
        )

    if len(bands) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

        ax = axes[0]
        mask = demod.decay_mask
        ax.semilogy(demod.t_mid[mask], demod.amplitude_corrected[mask], ".", ms=5,
                    label="Measured (floor-corrected)")
        if nl.tau0 is not None:
            ax.semilogy(nl.t_fit, nl.model_amplitude, "r-", linewidth=1.5,
                        label=r"Model $1/\tau(A) = 1/\tau_0 + \beta A$")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (cycles)")
        ax.set_title("Decay and the fitted amplitude law")
        ax.legend()
        ax.grid(True, alpha=0.3)

        ax = axes[1]
        amps = np.array([b.amplitude_mid for b in bands])
        qs = np.array([b.Q for b in bands])
        ax.loglog(amps, qs, "o", label="Banded local Q (measured)")
        if nl.tau0 is not None and len(amps) > 0:
            a_grid = np.geomspace(amps.min() / 1.5, amps.max() * 1.5, 100)
            ax.loglog(a_grid, nl.q_at(a_grid), "r-", linewidth=1.5, label="Model Q(A)")
        if r["Q_demod"] is not None:
            ax.axhline(r["Q_demod"], color="0.4", linestyle="--", linewidth=1,
                       label=f"Whole-decay Q_demod = {r['Q_demod']:.3e}")
        # Keep the measured points in view: the fitted law can diverge outside
        # the measured amplitude range (1/tau(A) -> 0 for negative beta).
        ax.set_ylim(0.5 * qs.min(), 2.0 * qs.max())
        ax.set_xlabel("Amplitude (cycles)")
        ax.set_ylabel("Local Q")
        ax.set_title("Q versus amplitude")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

### Next steps

- Many records at once: [`0.2_batch-analysis.ipynb`](0.2_batch-analysis.ipynb)
- Loading zipped Moku:Pro exports (`.csv.zip`): [`0.5_odin-phasemeter-data.ipynb`](0.5_odin-phasemeter-data.ipynb)
- What the coherent estimators do when they are *not* gated: [`0.3_profile-likelihood-q.ipynb`](0.3_profile-likelihood-q.ipynb) and [`0.4_frequency-estimation.ipynb`](0.4_frequency-estimation.ipynb)
- Why a white-noise bound does not describe these records: [`0.6_monte-carlo-crlb.ipynb`](0.6_monte-carlo-crlb.ipynb) § 5
- The estimator validated against pathological synthetics: [`20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb`](20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb)